In [1]:
%load_ext autoreload
%autoreload 2
%load_ext rpy2.ipython

In [2]:
import pandas as pd

import src
from src.load import DataLoader

r_colormap = src.r_colormap
r_out = str(src.OUT)
pd.options.display.float_format = "{:.1f}".format

In [3]:
%%R -i r_colormap -i r_out

suppressMessages(library(tidyverse))
library(ggplot2)
library(ggeffects)
library(here)
library(ggpubr)

options(scipen = 999)

cmap <- setNames(r_colormap$color, r_colormap$channel)

here() starts at /mnt/hdd/git/ytpop


# Load Data

In [4]:
dl = DataLoader()

videos = (
    dl.channels()
    .join(dl.videos(filtered=True, _ignore_sentence_filter=True), "channel_id")
    .to_pandas()
)
sents = dl.sentences(filtered=True).join(dl.popbert(filtered=True), "sentence_id").to_pandas()

sents = sents.groupby("video_id", observed=True).agg(
    n_sents=("video_id", "size"),
    n_elite=("elite", "sum"),
    n_pplcentr=("pplcentr", "sum"),
    avg_elite=("elite", "mean"),
    avg_pplcentr=("pplcentr", "mean"),
)

/nix/store/299jaglw5pjxd3mrf780a3v9578sz1vz-python3-3.11.10-env/lib/python3.11/site-packages/ibis/expr/types/relations.py:685: FutureWarning: Selecting/filtering arbitrary expressions in `Table.__getitem__` is deprecated and will be removed in version 10.0. Please use `Table.select` or `Table.filter` instead.
  warnings.warn(


# Dataset Summary Table

In [29]:
duration_overview = (
    (
        videos.merge(sents, on="video_id", how="left")
        .groupby(["channel", "video_was_live"], observed=True)
        .agg(avg_duration=("video_duration", "mean"))
    )
    .reset_index()
    .pivot(index="channel", columns="video_was_live", values="avg_duration")
)

duration_overview.columns = ["avg_duration_video", "avg_duration_livestream"]

channel_overview = (
    videos.merge(sents, on="video_id", how="left")
    .groupby("channel", observed=True)
    .agg(
        ch_followers=("channel_follower_count", "first"),
        videos=("video_was_live", lambda x: (x == 0).sum()),
        livestreams=("video_was_live", "sum"),
        avg_views=("video_view_count", "mean"),
        avg_likes=("video_like_count", "mean"),
        n_disabled_likes=("video_like_count", lambda x: x.isna().sum()),
        avg_duration=("video_duration", "mean"),
        n_sentences=("n_sents", "sum"),
        # avg_comments=("video_comment_count", lambda x: x.dropna().mean()),
        first_video=("video_datetime_upload", "min"),
        latest_video=("video_datetime_upload", "max"),
    )
)

channel_overview = channel_overview.join(duration_overview)

In [30]:
channel_overview

,ch_followers,videos,livestreams,avg_views,avg_likes,n_disabled_likes,avg_duration,n_sentences,first_video,latest_video,avg_duration_video,avg_duration_livestream
channel,,,,,,,,,,,,
AfD BT,515000,6208,527,54882.0,4408.3,0,1671.4,348122.0,2017-12-06 13:23:54,2025-02-04 18:00:08,409.0,16542.4
AfD TV,320000,1956,156,56700.7,4422.3,113,1771.5,163232.0,2017-12-08 00:21:22,2025-02-03 15:19:09,520.0,17462.8
CDU,28400,955,239,16771.1,136.7,2,1693.0,80763.0,2017-12-11 16:28:36,2025-02-04 22:08:58,498.1,6467.5
CSU,6340,178,250,9989.5,40.9,4,2750.8,12653.0,2017-12-14 21:19:06,2025-02-04 04:28:14,420.0,4410.3
FDP,26900,847,117,25497.8,82.0,956,1064.5,53062.0,2018-01-06 16:02:32,2025-02-04 17:56:50,475.4,5329.0
Greens,32900,634,91,13708.4,155.0,6,1098.4,49270.0,2018-01-12 10:16:37,2025-02-04 20:52:23,601.2,4562.0
Left,72900,770,408,12058.8,666.3,6,1505.4,134721.0,2017-12-11 14:33:45,2025-02-04 18:06:03,548.9,3310.5
SPD,31600,1106,213,8133.5,151.9,2,1776.5,158902.0,2017-12-07 12:17:56,2025-02-03 11:34:41,571.6,8033.2


In [32]:
# sum of durations

sum_of_seconds = videos.video_duration.sum()
print(f"Total sum of video durations: {round(sum_of_seconds / 60 / 60, 2)} hours")

Total sum of video durations: 6704.5 hours


In [33]:
# number of videos

count_videos = videos.video_id.size
print(f"Total number of valid videos: {count_videos}")

Total number of valid videos: 14655


In [34]:
# number of sentencs

count_sents = channel_overview.n_sentences.sum()
print(f"Total number of valid sentences: {count_sents}")

Total number of valid sentences: 1000725.0


In [35]:
summary_table = channel_overview.drop(["first_video", "latest_video"], axis=1).T

summary_table

channel,AfD BT,AfD TV,CDU,CSU,FDP,Greens,Left,SPD
ch_followers,515000.0,320000.0,28400.0,6340.0,26900.0,32900.0,72900.0,31600.0
videos,6208.0,1956.0,955.0,178.0,847.0,634.0,770.0,1106.0
livestreams,527.0,156.0,239.0,250.0,117.0,91.0,408.0,213.0
avg_views,54882.0,56700.7,16771.1,9989.5,25497.8,13708.4,12058.8,8133.5
avg_likes,4408.3,4422.3,136.7,40.9,82.0,155.0,666.3,151.9
n_disabled_likes,0.0,113.0,2.0,4.0,956.0,6.0,6.0,2.0
avg_duration,1671.4,1771.5,1693.0,2750.8,1064.5,1098.4,1505.4,1776.5
n_sentences,348122.0,163232.0,80763.0,12653.0,53062.0,49270.0,134721.0,158902.0
avg_duration_video,409.0,520.0,498.1,420.0,475.4,601.2,548.9,571.6
avg_duration_livestream,16542.4,17462.8,6467.5,4410.3,5329.0,4562.0,3310.5,8033.2


In [37]:
path = src.OUT / "tables/dataset_summary.csv"
path.unlink(missing_ok=True)
summary_table.to_csv(path)

# View Count Violin Plot

In [38]:
df = videos.merge(sents, on="video_id")

In [ ]:
%%R -i df -w 1000 -h 600

df_plot <- df %>%
   mutate(
      likes = video_like_count + 1,
      views = video_view_count + 1,
)

view_plot = ggplot(df_plot, aes(x=channel, y=views, fill=channel)) +
   geom_boxplot(alpha=0.6) +
   geom_violin(alpha=0.3, trim=T, scale="width") +
   scale_y_continuous(trans="log10", breaks=scales::breaks_log(n=5)) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 21
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(ViewCount)")

like_plot = ggplot(df_plot, aes(x=channel, y=likes, fill=channel)) +
   geom_boxplot(alpha=0.6) +
   geom_violin(alpha=0.3, trim=T, scale="width") +
   scale_y_continuous(trans="log10", breaks=scales::breaks_log(n=8)) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 21
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("log10(LikeCount)")


ggarrange(view_plot, like_plot, ncol=2)

p <- here(r_out, "/figures/view_count.svg")
if (file.exists(p)) file.remove(p)
ggsave(p)

Saving 13.9 x 8.33 in image


In addition: Warning messages:
1: Removed 760 rows containing non-finite outside the scale range
(`stat_boxplot()`). 
2: Removed 760 rows containing non-finite outside the scale range
(`stat_ydensity()`). 
3: Groups with fewer than two datapoints have been dropped.
ℹ Set `drop = FALSE` to consider such groups for position adjustment purposes. 


# Populism Amount Plot

In [ ]:
%%R -i df -w 1000 -h 600

df_plot <- df %>%
   mutate(
      elite = (n_elite / n_sents * 100) + 1,
      pplcentr = (n_pplcentr / n_sents * 100) + 1,
)

elite_plot = ggplot(df_plot, aes(x=channel, y=elite, fill=channel)) +
   geom_boxplot(alpha=0.6, outliers=F, coef=0.5) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("% Anti-Elitism")

pplcentr_plot = ggplot(df_plot, aes(x=channel, y=pplcentr, fill=channel)) +
   geom_boxplot(alpha=0.6, outliers=F, coef=0.5) +
   scale_color_manual(values=cmap, aesthetics=c("color", "fill")) +
   theme_ggeffects(
      base_family = "serif",
      base_size = 22
   ) +
   theme(
      axis.text.x=element_text(angle=20, hjust=1),
      legend.position = "none"
   ) +
   xlab("Channel") +
   ylab("% People-Centrism")


ggarrange(elite_plot, pplcentr_plot, ncol=2)

p <- here(r_out, "/figures/populism_per_party.svg")
if (file.exists(p)) file.remove(p)
ggsave(p)

Saving 13.9 x 8.33 in image
